
# SCB Silver Layer, Workforce Data

Transforms raw SCB JSON from the bronze volume into clean, typed Delta tables.

**Source**: `labour_market_platform.bronze_work_scb.landing/scb` (external volume)  
**Target**: `labour_market_platform.silver_work_scb` (Delta tables)

| Silver Table | Source File(s) | Description |
|---|---|---|
| `ssyk_mapping` | `ssyk_2012_mapping.json` | SSYK 2012 occupation code to name reference |
| `wages` | `wages_*.json` | Wage distribution by sector, occupation, gender, year |
| `aku_employment` | `aku_employment_stock_occupation.json` | Employment stock by occupation, gender, year |
| `aku_population` | `aku_population_region_labourstatus.json` | Population by region, labour status, gender, year |
| `aku_unemployment` | `aku_unemployed_age_sex.json` | Unemployment by age group, gender, year |

**Transformations**: 
* Flatten nested JSON → typed columns
* Standardize and alias column names
* Cast metrics to `DOUBLE`
* Handle suppressed values (`..`, `...`, `.`, etc.) as `NULL`
* Enforce `snake_case` naming conventions
* Deduplicate overlapping partitions
* Gender values mapped: `1` → men, `2` → women, `1+2` → total
* Sector codes mapped to labels (e.g. `1` → central government, `4-5` → private sector)
* Attachment codes mapped (e.g. `ANSTTOT` → employees total, `FA` → permanent employees)
* Employment occupation codes mapped to SSYK major group names (1-digit level)
* Region codes mapped to Swedish county names (e.g. `01` → Stockholm, `17` → Värmland)
* Labour status codes mapped (e.g. `ALÖS` → unemployed, `SYS` → employed)
* Unemployment age groups filtered to youth bracket (`15-24`) and ILO total (`tot15-74`), classified by `age_group_type`

In [0]:
import json, os, re
from glob import glob
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import functions as F

# Define Medallion Architecture paths
CATALOG       = "labour_market_platform"
BRONZE_SCHEMA = "bronze_work_scb"
SILVER_SCHEMA = "silver_work_scb"
VOLUME        = "landing"
BRONZE_ROOT   = "scb"
BASE_PATH     = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}/{BRONZE_ROOT}"

# Ensure Silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

def get_latest_partition(subfolder: str) -> str:
    """Finds the most recent data drop in the Bronze layer."""
    parts = sorted(glob(f"{BASE_PATH}/{subfolder}/ingestion_date=*"))
    if not parts:
        raise FileNotFoundError(f"No partitions found under {BASE_PATH}/{subfolder}/")
    return parts[-1]

print(f"Bronze root:{BASE_PATH}")
print(f"Latest wages partition: {get_latest_partition('wages')}")
print(f"Latest AKU partition: {get_latest_partition('aku')}")

In [0]:
from pyspark.sql import DataFrame
def parse_scb_to_spark(file_paths: list, column_renames: dict = None) -> DataFrame:

    """
    Parses complex SCB JSON files into a flattened, typed PySpark DataFrame.
    """
    column_renames = column_renames or {}
    all_rows = []
    schema_fields = {} 

    # Convert names to snake_case, remove special characters and replace spaces
    def to_snake_case(s: str) -> str:
        return re.sub(r'[^a-z0-9]+', '_', s.lower().strip()).strip('_')

    # SCB uses these symbols for missing/suppressed data
    suppressed_vals = {'..', '...', '.', ''}

    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            raw = json.load(f)

        cols_meta = raw.get('columns', [])

        # Dynamically build the PySpark Schema
        for c in cols_meta:
            raw_name = to_snake_case(c['text'])
            final_name = column_renames.get(raw_name, raw_name)
            
            if final_name not in schema_fields:
                # Dimensions ('d') and Time ('t') are Strings. Metrics ('c') are Doubles.
                c_type = StringType() if c['type'] in ('d', 't') else DoubleType()
                schema_fields[final_name] = StructField(final_name, c_type, True)

        # Extract column names in order
        dim_names = [column_renames.get(to_snake_case(c['text']), to_snake_case(c['text'])) for c in cols_meta if c['type'] in ('d', 't')]
        val_names = [column_renames.get(to_snake_case(c['text']), to_snake_case(c['text'])) for c in cols_meta if c['type'] == 'c']

        # Flatten the nested data
        for row in raw.get('data', []):
            record = {}
            for name, val in zip(dim_names, row['key']):
                record[name] = val
            for name, val in zip(val_names, row['values']):
                # Standardize missing data to native SQL NULLs
                record[name] = None if val in suppressed_vals else float(val)
            all_rows.append(record)

    # Create a single PySpark DataFrame using the explicit schema
    unified_schema = StructType(list(schema_fields.values()))
    return spark.createDataFrame(all_rows, schema=unified_schema)

In [0]:
def save_to_silver(df, table_name: str, dedupe_columns: list):
    """
    Deduplicates the PySpark DataFrame and writes it to a Delta table.
    """
    # Deduplicate based on the primary keys
    df_clean = df.dropDuplicates(dedupe_columns)
    
    target_path = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    
    # Save as a Delta table
    df_clean.write.format("delta").mode("overwrite").saveAsTable(target_path)
    
    print(f"Successfully wrote {df_clean.count()} rows to {target_path}")

In [0]:
# Read SSYK 2012 mapping from the bronze layer (ingested from SCB API)
ssyk_path = f"{BASE_PATH}/reference/ingestion_date=*/ssyk_2012_mapping.json"
ssyk_files = sorted(glob(ssyk_path))

if not ssyk_files:
    raise FileNotFoundError(f"No SSYK mapping found. Run the bronze ingestion notebook first.")

with open(ssyk_files[-1], 'r', encoding='utf-8') as f:
    ssyk_metadata = json.load(f)

# Extract the SSYK 2012 variable (coded as 'Yrke2012' in SCB metadata)
ssyk_meta = next(var for var in ssyk_metadata["variables"] if var["code"] == "Yrke2012")
codes = ssyk_meta["values"]
texts = ssyk_meta["valueTexts"]

# Build DataFrame and clean text labels (SCB prepends the code in the text)
df_ssyk = spark.createDataFrame(list(zip(codes, texts)), ["occupation_code", "raw_occupation_name"])
df_ssyk = df_ssyk.withColumn(
    "occupation_name",
    F.trim(F.regexp_replace(F.col("raw_occupation_name"), r"^\d+\s+", ""))
).drop("raw_occupation_name")

# Save as a reference table in the Silver layer
target_table = f"{CATALOG}.{SILVER_SCHEMA}.ssyk_mapping"
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
df_ssyk.write.mode("overwrite").saveAsTable(target_table)

print(f"Successfully saved {df_ssyk.count()} SSYK codes to {target_table}")

In [0]:
# PROCESS WAGES
WAGES_RENAMES = {
    "occupation_ssyk_2012": "occupation_code",
    "sex": "gender",
    
    # Base Salary Metrics
    "monthly_salary":   "monthly_salary_avg",
    
    # Percentiles adding salary and p as percentile suffix
    "10th_percentile":  "salary_p10",
    "25th_percentile":  "salary_p25",
    "75th_percentile":  "salary_p75",
    "90th_percentile":  "salary_p90",
    
    # Confidence Intervals 95% shortened to "ci95"
    "monthly_salary_95_percent_confidence_interval":    "monthly_salary_avg_ci95",
    "median_95_percent_confidence_interval":    "monthly_salary_median_ci95",
    "average_monthly_salary_10th_percentile_95_percent_confidence_interval":    "salary_p10_ci95",
    "average_monthly_salary_25th_percentile_95_percent_confidence_interval":    "salary_p25_ci95",
    "average_monthly_salary_75th_percentile_95_percent_confidence_interval":    "salary_p75_ci95",
    "average_monthly_salary_90th_percentile_95_percent_confidence_interval":    "salary_p90_ci95",
}

# Glob catches all wage files, e.g. wages_(date).json
wages_files = glob(f"{get_latest_partition('wages')}/*.json")
df_wages = parse_scb_to_spark(wages_files, WAGES_RENAMES)

# Map coded values to readable labels
df_wages = df_wages.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_wages = df_wages.replace({
    "0": "all sectors",
    "1": "central government",
    "2": "municipalities",
    "3": "county council",
    "1-3": "public sector",
    "4": "manual workers private",
    "5": "non-manual workers private",
    "4-5": "private sector",
}, subset=["sector"])

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.wages")
save_to_silver(df_wages, "wages", dedupe_columns=["sector", "occupation_code", "gender", "year"])

In [0]:
# PROCESS AKU EMPLOYMENT
EMP_RENAMES = {
    "degree_of_attachment_to_the_labour_market": "attachment_code",
    "occupation":   "occupation_code",
    "sex": "gender",
    "employed_persons": "employed_thousands",
    "margin_of_error":  "moe_thousands",
}

emp_files = glob(f"{get_latest_partition('aku')}/aku_employment_stock_occupation*.json")
df_emp = parse_scb_to_spark(emp_files, EMP_RENAMES)

# Map coded values to readable labels
df_emp = df_emp.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_emp = df_emp.replace({
    "ANSTTOT": "employees total",
    "FA": "permanent employees",
    "SYSTOT": "total employment",
}, subset=["attachment_code"])
df_emp = df_emp.replace({
    "0000": "all occupations",
    "0": "armed forces",
    "0002": "unidentifiable",
    "1": "managers",
    "2": "advanced higher education",
    "3": "higher education",
    "4": "administration and customer service",
    "5": "service, care and sales",
    "6": "agricultural and forestry",
    "7": "building and manufacturing",
    "8": "mechanical manufacturing and transport",
    "9": "elementary occupations",
}, subset=["occupation_code"])

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.aku_employment")
save_to_silver(df_emp, "aku_employment", dedupe_columns=["attachment_code", "occupation_code", "gender", "year"])

In [0]:
# PROCESS AKU POPULATION
POP_RENAMES = {
    "labour_status":     "labour_status_code",
    "sex": "gender",
    "thousands":              "pop_thousands",
    "margin_of_error_1000s":  "moe_thousands",
    "percent":                  "pop_percent",
    "margin_of_error_percent":  "moe_percent",
}

pop_files = glob(f"{get_latest_partition('aku')}/aku_population_region_labourstatus*.json")
df_pop = parse_scb_to_spark(pop_files, POP_RENAMES)

# Map coded values to readable labels
df_pop = df_pop.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_pop = df_pop.replace({
    "00": "Sweden",
    "0050": "Sweden excl. Stockholm, Göteborg, Malmö",
    "01": "Stockholm",
    "0180": "Stockholm",   
    "03": "Uppsala",
    "04": "Södermanland",
    "05": "Östergötland",
    "06": "Jönköping",
    "07": "Kronoberg",
    "08": "Kalmar",
    "09": "Gotland",
    "10": "Blekinge",
    "12": "Skåne",
    "1280": "Skåne",
    "13": "Halland",
    "14": "Västra Götaland",
    "1480": "Västra Götaland",
    "17": "Värmland",
    "18": "Örebro",
    "19": "Västmanland",
    "20": "Dalarna",
    "21": "Gävleborg",
    "22": "Västernorrland",
    "23": "Jämtland",
    "24": "Västerbotten",
    "25": "Norrbotten",
}, subset=["region"])
df_pop = df_pop.replace({
    "TOTALT": "total",
    "ALÖS": "unemployed",
    "EIAKR": "not in labour force",
    "SYS": "employed",
}, subset=["labour_status_code"])

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.aku_population")
save_to_silver(df_pop, "aku_population", dedupe_columns=["region", "labour_status_code", "gender", "year"])

In [0]:
# PROCESS AKU UNEMPLOYMENT
UNEMP_RENAMES = {
    "labour_status":        "labour_status_code",
    "age":                           "age_group",
    "sex": "gender",
    "1000s":                   "unemp_thousands",
    "margin_of_error_1000s":     "moe_thousands",
    "percent":                   "unemp_percent",
    "margin_of_error_percent":     "moe_percent",
}

unemp_files = glob(f"{get_latest_partition('aku')}/aku_unemployed_age_sex*.json")
df_unemp = parse_scb_to_spark(unemp_files, UNEMP_RENAMES)

# Map coded values to readable labels
df_unemp = df_unemp.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_unemp = df_unemp.replace({
    "ALÖS": "unemployed",
    "HELTSTUDSOK": "unemployed full-time students",
}, subset=["labour_status_code"])

# Keep only youth bracket (15-24) and ILO standard total (tot15-74), drop redundant overlapping totals
df_unemp = df_unemp.filter(F.col("age_group").isin("15-24", "tot15-74"))

# Classify age groups
df_unemp = df_unemp.withColumn(
    "age_group_type",
    F.when(F.col("age_group").startswith("tot"), F.lit("total")).otherwise(F.lit("youth"))
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.aku_unemployment")
save_to_silver(df_unemp, "aku_unemployment", dedupe_columns=["labour_status_code", "age_group", "gender", "year"])

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.ssyk_mapping LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.wages LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_employment LIMIT 10

In [0]:
%sql
SELECT DISTINCT region FROM labour_market_platform.silver_work_scb.aku_population

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_population WHERE Region NOT IN ("Sweden", "Sweden excl. Stockholm, Göteborg, Malmö") LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_unemployment LIMIT 10